In [1]:
import sys
import os
# sys.path.append('./zennit-crp')  # Adjust to the correct relative path
# os.chdir("./zennit-crp")  # Change to the correct directory

import torch
# from torchvision.models.vgg import vgg16_bn, VGG16_BN_Weights
import torchvision.transforms as T
from PIL import Image
from zennit.canonizers import SequentialMergeBatchNorm
from zennit.composites import EpsilonPlusFlat

import torchvision
from crp.concepts import ChannelConcept
from crp.helper import get_layer_names
from crp.attribution import CondAttribution
from crp.visualization import FeatureVisualization
from crp.image import imgify, plot_grid
from tutorials.VGG16_ImageNet.download_imagenet import download


In [2]:
class ChannelConceptMax(ChannelConcept):
    def attribute(self, relevance, mask=None, layer_name: str | None = None, abs_norm=True):
        if isinstance(mask, torch.Tensor):
            relevance = relevance * mask

        rel_l = torch.max(relevance.view(*relevance.shape[:2], -1), dim=-1).values

        if abs_norm:
            rel_l = rel_l / (torch.abs(rel_l).sum(-1).view(-1, 1) + 1e-10)

        return rel_l

In [3]:
from pathlib import Path
# import mlflow
from src.modeling.decode_head.binary_classifier import BinaryClassifier
import albumentations



device = "cuda:0" if torch.cuda.is_available() else "cpu"

# mlflow.set_tracking_uri('http://mlflow.rationai-mlflow:5000/')
# logged_model = 'runs:/c38a483305574912a2b3f15bc64a2284/model/MLFlowModelCheckpoint/vgg16_prostate_best'

# # Load model as a PyFuncModel.
# model = mlflow.pyfunc.load_model(logged_model)
# model = vgg16_bn(weights=VGG16_BN_Weights.IMAGENET1K_V1)

model_state_dict = torch.load("state_dict_version1_tl_best.pth", map_location=device)
model_features = torchvision.models.vgg16().features
model_features.load_state_dict(model_state_dict['features'])
model_decoder = BinaryClassifier(in_features=512)
model_decoder.load_state_dict(model_state_dict['decode_head'])

class VGG16_Rationai(torch.nn.Module):
    def __init__(self, features, decode_head):
        super(VGG16_Rationai, self).__init__()
        self.features = features
        self.decode_head = decode_head

    def forward(self, x):
        x = self.features(x)
        x = self.decode_head(x)
        return x

model = VGG16_Rationai(
    features=model_features,
    decode_head=model_decoder
)
model = model.to(device)

model.eval()

layer_names = get_layer_names(model, [torch.nn.Conv2d, torch.nn.Linear])

attribution = CondAttribution(model)

canonizers = [SequentialMergeBatchNorm()]
composite = EpsilonPlusFlat(canonizers)

# Original code for VGG16_bn
# # separate normalization from resizing for plotting purposes later
# transform = T.Compose([T.Resize(256), T.CenterCrop(224), T.ToTensor()])
# preprocessing =  T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

# transform_norm = T.Compose([
#     transform,
#     preprocessing
# ])

transform_norm=albumentations.Normalize(
    mean=[230.6765, 187.0875, 222.7069],
    std=[23.1706, 45.7364, 22.656]
)



# data_path = Path("tutorials/ImageNet_data")

# if len(list(data_path.iterdir())) > 0:
#     print(f"Using existing data at {data_path}")
# else:  # download ImageNet validation set
#     download(data_path)

# # apply no normalization here!
# imagenet_data = torchvision.datasets.ImageNet(data_path, transform=transform, split="val")

In [4]:
import hydra
import mlflow
from omegaconf import DictConfig, OmegaConf
import os

os.environ["HYDRA_FULL_ERROR"] = "1"  # to see full error messages in case of errors

# load the default config from the config directory
with hydra.initialize(version_base=None, config_path="conf"):
        cfg = hydra.compose(config_name="crp-prov-gigapath", overrides=[])

mlflow.set_tracking_uri("http://mlflow.rationai-mlflow:5000/")

# instantiate the data module using the default config
data_module = hydra.utils.instantiate(
    cfg.data,
    _recursive_=True,
    _convert_="partial",
)

# uris = ["mlflow-artifacts:/65/7382f77c8a404d479d5b070146d62f86/artifacts/Prostate - test"]
# dataset = ProstateCancer(
#     uris=uris,
#     transforms=albumentations.Normalize(
#         mean=[230.6765, 187.0875, 222.7069],
#         std=[23.1706, 45.7364, 22.656]
#     )
# )
# data_module = DataModule(
#     dataset=dataset,
#     batch_size=4,
#     num_workers=2,
# )



In [ ]:
dl = data_module.test_dataloader()

class ProstateWrapper(torch.utils.data.Dataset):
    """This wrapper just removes metadata from the (X, Y, metadata) triple that the RationAI data module returns.
    It returns only the (X, Y) pair.
    """
    def __init__(self, dataset):
        self.dataset = dataset

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        x, y, _ = self.dataset[idx]
        return x, y

print(len(dl))
print(type(dl.dataset[166][1]))

1965
<class 'torch.Tensor'>


In [6]:
display(layer_names)

print(type(dl))

['features.0',
 'features.2',
 'features.5',
 'features.7',
 'features.10',
 'features.12',
 'features.14',
 'features.17',
 'features.19',
 'features.21',
 'features.24',
 'features.26',
 'features.28',
 'decode_head.proj']

<class 'torch.utils.data.dataloader.DataLoader'>


In [7]:
# ORIGINAL CODE for VGG16_BN
# def get_ith_last_feature_layer(i):
#     # get the ith last feature layer
#     return layer_names[-4 - i]

def get_ith_last_feature_layer(i):
    # get the ith last feature layer
    return layer_names[-2 - i]

In [8]:
# ORIGINAL CODE for VGG16_BN
# assert get_ith_last_feature_layer(0) == "features.40"
# assert get_ith_last_feature_layer(1) == "features.37"

assert get_ith_last_feature_layer(0) == "features.28", "Expected features.28, got " + get_ith_last_feature_layer(0)
assert get_ith_last_feature_layer(1) == "features.26", "Expected features.26, got " + get_ith_last_feature_layer(1)

In [20]:
# ORIGINAL CODE for VGG16_BN
# fv_path = "tutorials/VGG16_ImageNet"

# def feature_visualization(concept, layer_names):
#     fv = FeatureVisualization(attribution, imagenet_data, { name: concept for name in layer_names }, preprocess_fn=preprocessing, path=fv_path)
#     return fv

fv_path = "tutorials/VGG16_Prostate"
def feature_visualization(concept, layer_names):
    fv = FeatureVisualization(attribution, ProstateWrapper(dataset=dl.dataset), { name: concept for name in layer_names }, path=fv_path)
    return fv


In [10]:
softmax = torch.nn.Softmax(dim=-1)

def get_ids(sample, prob=0.05):
    y = model(sample)
    probs = softmax(y).squeeze()  # shape: (num_classes,)
    mask = probs >= prob
    selected_ids = mask.nonzero(as_tuple=True)[0]
    selected_probs = probs[selected_ids]

    # Sort by probability in descending order
    sorted_indices = torch.argsort(selected_probs, descending=True)
    sorted_ids = selected_ids[sorted_indices]
    sorted_probs = selected_probs[sorted_indices]

    return sorted_ids.tolist(), sorted_probs.tolist()


In [11]:
# ORIGINAL CODE for ImageNet dataset
# def get_label(id):
#     return imagenet_data.classes[id]

def get_label(id):
    return "positive" if id == 1 else "negative" if id == 0 else "heck! there's a problem somewhere"

In [12]:
# ORIGINAL CODE for ImageNet dataset
# def get_image(path):
#     image = Image.open(path)
#     sample = transform_norm(image).unsqueeze(0).to(device)

#     # zennit requires gradients
#     sample.requires_grad = True
    
#     return image, sample

In [13]:
def get_conditions(y, channels_sequence):
    channels_names = [get_ith_last_feature_layer(i) for i in range(len(channels_sequence))]
    channels = {name: [id]  for name, id in zip(channels_names, channels_sequence)}
    conditions = {'y' : [y], **channels}

    return conditions

In [14]:
def get_top_concepts(sample, y, channels_sequence, concept_attribution, num_of_concepts):
    attr = attribution(sample, [get_conditions(y, channels_sequence)], composite, record_layer=layer_names)

    rel_c = concept_attribution.attribute(attr.relevances[get_ith_last_feature_layer(len(channels_sequence))], abs_norm=True)

    rel_values_tensor, concept_ids_tensor = torch.topk(rel_c[0], num_of_concepts)
    concept_ids = [int(id) for id in concept_ids_tensor]
    rel_values = [float(value) * 100 for value in rel_values_tensor]

    return concept_ids, rel_values

In [15]:
def show_top_concepts(sample, concept_ids,  y, channels_sequence):
    conditions =  get_conditions(y, channels_sequence)
    last_layer = get_ith_last_feature_layer(len(channels_sequence))
    new_conditions = [{**conditions, last_layer: [id]} for id in concept_ids]
    heatmap, _, _, _ = attribution(sample, new_conditions, composite)
    display(imgify(heatmap, symmetric=True, grid=(1, len(concept_ids))))

In [16]:
def show_top_representatives(feature_visualization, concept_ids, layer, num_of_representatives):
    ref_c = feature_visualization.get_max_reference(concept_ids, layer, "relevance", (0, num_of_representatives), composite=composite, plot_fn=None)

    for id, images in zip(concept_ids, ref_c.values()):
        print(f"Concept {id}")
        display(imgify(images[0], grid=(1, num_of_representatives)))
        display(imgify(images[1], symmetric=True, grid=(1, num_of_representatives)))

In [17]:
def run(sample, y, channels_sequence, concept_attribution, feature_visualization, num_of_concepts, num_of_representatives):
    print(f"Predicted label {get_label(y)} (id: {y})")
    layer = get_ith_last_feature_layer(len(channels_sequence))
    print(f'Showing layer {layer}')

    concept_ids, rel_values = get_top_concepts(sample, y, channels_sequence, concept_attribution, num_of_concepts)
    
    print(f"Top {num_of_concepts} concepts are {concept_ids} with relevance {rel_values}")

    show_top_concepts(sample, concept_ids, y, channels_sequence)

    show_top_representatives(feature_visualization, concept_ids, layer, num_of_representatives)


In [18]:
print(layer_names)

['features.0', 'features.2', 'features.5', 'features.7', 'features.10', 'features.12', 'features.14', 'features.17', 'features.19', 'features.21', 'features.24', 'features.26', 'features.28', 'decode_head.proj']


In [21]:
# Original code for IMAGE_NET dataset
# feature_visualization(ChannelConcept(), [get_ith_last_feature_layer(i) for i in [2, 3, 4]]).run(composite,  0, len(imagenet_data), 32, 100)

feature_visualization(ChannelConcept(), [get_ith_last_feature_layer(i) for i in [2, 3, 4]]).run(composite,  0, len(dl.dataset), 32, 100)


Running Analysis...


  0%|          | 0/2948 [00:00<?, ?it/s]

/home/jovyan/crp/crp/visualization.py:121: FutureWarning: The input object of type 'Tensor' is an array-like implementing one of the corresponding protocols (`__array__`, `__array_interface__` or `__array_struct__`); but not a sequence (or 0-D). In the future, this object will be coerced as if it was first converted using `np.array(obj)`. To retain the old behaviour, you have to either modify the type 'Tensor', or assign to an empty array created with `np.empty(correct_shape, dtype=object)`.
  targets_samples = np.array(targets_samples)  # numpy operation needed
/home/jovyan/crp/crp/visualization.py:121: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  targets_samples = np.array(targets_samples)  # numpy operation needed


TypeError: can't convert np.ndarray of type numpy.object_. The only supported types are: float64, float32, float16, complex64, complex128, int64, int32, int16, int8, uint64, uint32, uint16, uint8, and bool.

In [ ]:
concept_sum = ChannelConcept()
concept_max = ChannelConceptMax()
fv_sum = feature_visualization(concept_sum, layer_names)
fv_max = feature_visualization(concept_max, layer_names)

In [19]:
image, sample = get_image("tutorials/images/lizard.jpg")

display(image)

ids, probs = get_ids(sample)

display(ids, probs)

run(sample, ids[0], [], concept_sum, fv_sum, num_of_concepts=10, num_of_representatives=8)

In [ ]:
image, sample = get_image("tutorials/images/lizard.jpg")

display(image)

ids, probs = get_ids(sample)

display(ids, probs)

run(sample, ids[0], [], concept_max, fv_max, num_of_concepts=10, num_of_representatives=8)

In [ ]:
image, sample = get_image("tutorials/ImageNet_data/val/n07697313/ILSVRC2012_val_00008200.JPEG")
display(image)

ids, probs = get_ids(sample)

display(ids, probs)

run(sample, ids[0], [], concept_sum, fv_sum, num_of_concepts=10, num_of_representatives=8)

In [ ]:
image, sample = get_image("tutorials/ImageNet_data/val/n13133613/ILSVRC2012_val_00000412.JPEG")
run(sample, ids[0], [], concept_sum, fv_sum, num_of_concepts=10, num_of_representatives=8)

In [ ]:
image, sample = get_image("tutorials/ImageNet_data/val/n01775062/ILSVRC2012_val_00005290.JPEG")
display(image)

ids, probs = get_ids(sample)

display(ids, probs)

run(sample, ids[0], [], concept_sum, fv_sum, num_of_concepts=10, num_of_representatives=8)

In [ ]:
image, sample = get_image("tutorials/ImageNet_data/val/n01775062/ILSVRC2012_val_00040214.JPEG")
display(image)

ids, probs = get_ids(sample)

display(ids, probs)

run(sample, ids[0], [], concept_sum, fv_sum, num_of_concepts=10, num_of_representatives=8)

In [ ]:
image, sample = get_image("tutorials/ImageNet_data/val/n01775062/ILSVRC2012_val_00040214.JPEG")
display(image)

ids, probs = get_ids(sample)

# display(ids, probs)

run(sample, ids[0], [71], concept_sum, fv_sum, num_of_concepts=10, num_of_representatives=8)

In [ ]:
image, sample = get_image("tutorials/images/lizard.jpg")

display(image)

ids, probs = get_ids(sample)

display(ids, probs)

run(sample, ids[0], [], concept_sum, fv_sum, num_of_concepts=10, num_of_representatives=8)

In [ ]:
image, sample = get_image("tutorials/ImageNet_data/val/n13052670/ILSVRC2012_val_00008275.JPEG")
run(image, sample, concept_sum, fv_sum)